In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests, zipfile, io, os
import pandas as pd

In [ ]:
# Listings fetch
domain = "datasets.techmatrix.it/airml"
token = "DI_xeno_2026"

cities = ["sicilia", "trentino", "venezia", "roma", "puglia",
          "napoli", "firenze", "milano", "bergamo", "bologna"]

for city in cities:
    data_dir = os.path.join("./data", city)
    if os.path.exists(data_dir):
        print(f"Skipping {city}: folder already exists")
        continue
    
    url = f"https://{domain}/listings/{city}.zip?token={token}"
    resp = requests.get(url, stream=True)
    if resp.ok:
        os.makedirs(data_dir, exist_ok=True)
        zip_path = os.path.join("./data", f"{city}.zip")
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(8192):
                if chunk:
                    f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(path=data_dir)
        os.remove(zip_path)
    else:
        print(f"Failed to download {city}: {resp.status_code}")

In [ ]:
# # Reviews fetch

# for city in cities:
#     url = f"https://{domain}/reviews/{city}.zip?token={token}"
#     resp = requests.get(url, stream=True)
#     if resp.ok:
#         data_dir = os.path.join("./data", city)
#         if not os.path.exists(data_dir):
#             os.makedirs(data_dir, exist_ok=True)
#         zip_path = os.path.join("./data", f"{city}.zip")
#         with open(zip_path, "wb") as f:
#             for chunk in resp.iter_content(8192):
#                 if chunk:
#                     f.write(chunk)
#         with zipfile.ZipFile(zip_path, "r") as z:
#             z.extractall(path=data_dir)
#         os.remove(zip_path)
#     else:
#         print(f"Failed to download {city}: {resp.status_code}")

# Filtering

In [ ]:
COLS_TO_DROP = {
    # URL / immagini
    "listing_url", "picture_url", "host_thumbnail_url", "host_picture_url", "host_url",
    # Testuali / identificativi listing
    "name", "description", "neighborhood_overview", "calendar_updated",
    # Identificatori di scraping / metadati tecnici
    "scrape_id", "last_scraped", "source",
    # Identificatori personali / dati host
    "host_id", "host_name", "host_since", "host_location", "host_about",
    "host_neighbourhood", "host_listings_count", "host_total_listings_count",
    "host_verifications", "host_has_profile_pic", "host_identity_verified",
    # Metriche risposta host
    "host_response_time", "host_response_rate", "host_acceptance_rate", "host_is_superhost",
    # Location duplicate / non predittive
    "neighbourhood", "neighbourhood_group_cleansed",
    # Testo derivabile / calcolato
    "bathrooms_text", "first_review", "last_review",
    # Calcolati host (aggregati)
    "calculated_host_listings_count", "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms", "calculated_host_listings_count_shared_rooms",
}
dfs = []

for sub in os.listdir("./data"):
    subpath = os.path.join("./data", sub)
    if not os.path.isdir(subpath):
        continue
    csv_path = os.path.join(subpath, "listings.csv")
    if os.path.exists(csv_path):
        dfs.append(pd.read_csv(csv_path, usecols=lambda col: col not in COLS_TO_DROP))
    else:
        for root, _, files in os.walk(subpath):
            if "listings.csv" in files:
                dfs.append(pd.read_csv(os.path.join(root, "listings.csv"), usecols=lambda col: col not in COLS_TO_DROP))
                break

if dfs:
    listings = pd.concat(dfs, ignore_index=True)
else:
    listings = pd.DataFrame()

listings.info()

In [ ]:
# REVIEW_COLS_TO_DROP = {
#     "reviewer_name", "date"
# }
# review_dfs = []

# for sub in os.listdir("./data"):
#     subpath = os.path.join("./data", sub)
#     if not os.path.isdir(subpath):
#         continue
#     csv_path = os.path.join(subpath, "reviews.csv")
#     if os.path.exists(csv_path):
#         review_dfs.append(pd.read_csv(csv_path, usecols=lambda col: col not in REVIEW_COLS_TO_DROP))
#     else:
#         for root, _, files in os.walk(subpath):
#             if "reviews.csv" in files:
#                 review_dfs.append(pd.read_csv(os.path.join(root, "reviews.csv"), usecols=lambda col: col not in REVIEW_COLS_TO_DROP))
#                 break

# if review_dfs:
#     reviews = pd.concat(review_dfs, ignore_index=True)
# else:
#     reviews = pd.DataFrame()

# reviews.info()

In [ ]:
# 1. Drop righe duplicate
dupes = listings.duplicated().sum()
listings = listings.drop_duplicates().reset_index(drop=True)
print(f"Righe duplicate rimosse: {dupes}")

In [ ]:

# 2. Drop righe con target nullo (price) o con più del 70% di valori nulli
null_price = listings["price"].isna().sum()
listings = listings.dropna(subset=["price"])
print(f"Righe con price nullo rimosse: {null_price}")

# Righe con più del 70% di valori nulli
thresh = int(0.70 * listings.shape[1])
sparse_mask = listings.isna().sum(axis=1) > thresh
sparse_count = sparse_mask.sum()
listings = listings[~sparse_mask].reset_index(drop=True)
print(f"Righe con >70% nulli rimosse: {sparse_count}")

In [ ]:
# 3. Drop righe con accommodates < 1
low_acc = (listings["accommodates"] < 1).sum()
listings = listings[listings["accommodates"] >= 1].reset_index(drop=True)
print(f"Righe con accommodates < 1 rimosse: {low_acc}")

In [ ]:
# 4. Normalizza subito il target price a numerico
if "price" in listings.columns:
    listings["price"] = pd.to_numeric(
        listings["price"].astype(str).str.replace(r"[$,]", "", regex=True),
        errors="coerce"
    ).astype("Int64")
    print("Converted price to numeric dtype:", listings["price"].dtype)

## 2. Analisi Esplorativa dei Dati (EDA)

### 2.1 Statistiche Generali


Questa sottosezione calcola le statistiche descrittive per le colonne numeriche del dataframe `listings`.

Le celle sono suddivise in sotto-sezioni:

- **2.1.a** Elenco colonne numeriche (scopo: verificare quali feature numeriche sono disponibili).
- **2.1.b** Statistiche descrittive: media, std, quartili, conteggio valori mancanti e unici.
- **2.1.c** Top valori per alcune colonne categoriche..
- **2.1.d** Matrice di correlazione tra variabili numeriche e relativa heatmap.


#### 2.1.a Elenco colonne numeriche

In [ ]:
num_cols = listings.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numero colonne numeriche: {len(num_cols)}")
print(num_cols[:50])


Spiegazione delle principali colonne numeriche mostrate sopra:

- **price**: prezzo per notte. 
- **accommodates**: numero massimo di ospiti supportati.
- **bathrooms**: numero di bagni.
- **bedrooms**: numero di camere da letto.
- **beds**: numero di letti disponibili.
- **minimum_nights / maximum_nights**: vincoli min/max di soggiorno.
- **availability_30 / availability_60 / availability_90 / availability_365**: giorni disponibili nei rispettivi intervalli.
- **number_of_reviews, reviews_per_month**: conteggio recensioni e frequenza mensile.
- **review_scores_rating**: punteggio medio delle recensioni (aggregato).
- **estimated_occupancy_l365d**: occupazione stimata su ultimi 365 giorni (target Task B).
- **estimated_revenue_l365d**: ricavo stimato annuo (se presente).
- **latitude / longitude**: coordinate geografiche (possono servire per mappe o distanza dal centro).
- **n_amenities**: (se calcolata) numero di servizi offerti dall'alloggio.

#### 2.1.b Statistiche descrittive numeriche

In [ ]:
stats = listings[num_cols].describe().transpose()
stats['missing'] = listings[num_cols].isna().sum()
stats['unique'] = listings[num_cols].nunique()
display(stats)

Interpretazione e raccomandazioni sulle statistiche descrittive:

- **count**: numero di osservazioni non-nulle; colonne con `count` molto minore del dataset richiedono attenzione (imputazione o drop).
- **mean / std**: media e deviazione standard; confrontale per capire dispersione e presenza di outlier.
- **min / 25% / 50% / 75% / max**: quantili utili per individuare asimmetrie e outlier (max >> 75% + IQR indica outlier).
- **missing**: numero di valori mancanti (calcolato nella tabella salvata). Per `price`, questo valore include anche eventuali stringhe non convertibili che vengono trasformate in `NaN` durante il parsing.
- **unique**: numero di valori distinti; se basso può essere trattata come categorica.

In [ ]:
# 2.1.c Categorie top values
cat_cols = [c for c in ['room_type', 'property_type', 'neighbourhood_cleansed'] if c in listings.columns]
for c in cat_cols:
    print(f"\nTop values for {c}:")
    display(listings[c].value_counts(dropna=False).head(20))

In [ ]:
# 2.1.d Correlation matrix and heatmap
num_cols = listings.select_dtypes(include=[np.number]).columns.tolist()
corr = listings[num_cols].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation matrix (numeric features)')
plt.show()

Questa sottosezione visualizza la **matrice di correlazione** tra tutte le variabili numeriche del dataframe tramite una **heatmap**:

- Ogni cella della matrice misura la correlazione lineare tra due variabili numeriche.
- I valori vanno da **-1** a **1**:
  - **1** = correlazione positiva perfetta.
  - **0** = nessuna relazione lineare evidente.
  - **-1** = correlazione negativa perfetta.
- La diagonale principale è sempre pari a 1, perché ogni variabile è perfettamente correlata con sé stessa.

Come interpretare i colori della heatmap:

- Toni **rossi**: relazione positiva, cioè le due variabili tendono a crescere insieme.
- Toni **blu**: relazione negativa, cioè una cresce mentre l'altra tende a scendere.
- Toni molto chiari o quasi neutri: relazione debole o assente.

Cose interessanti da osservare nel nostro caso:

- Blocchi forti tra `accommodates`, `bathrooms`, `bedrooms` e `beds`: indicano che descrivono dimensioni simili dell'alloggio.
- Correlazioni tra `availability_30`, `availability_60`, `availability_90` e `availability_365`: sono attese perché misurano la disponibilità nello stesso senso ma su finestre diverse.
- Relazioni tra `number_of_reviews`, `reviews_per_month` e i punteggi recensione: utili per capire se l'attività dell'alloggio è associata alla qualità percepita.
- Eventuali correlazioni con `price` e `estimated_occupancy_l365d`: sono quelle più utili in vista dei modelli di regressione.

### 2.3 Specifiche per le singole task


Questa sezione contiene le specifiche per l'analisi esplorativa dei dati mirata alle singole task.

Le celle sono suddivise in sotto-sezioni:

- **2.3.a** Price Prediction
- **2.3.b** Occupancy Regression
- **2.3.c** NLP Classification
- **2.3.d** Recommendation System


#### 2.3.b - Occupancy Regression

Obiettivo: esplorare `estimated_occupancy_l365d` e le sue relazioni con le feature disponibili, escludendo le colonne che creerebbero leakage: `price`, `estimated_revenue_l365d`, tutte le colonne `availability_*` e le colonne `number_of_reviews*`.

Sottosezioni:

- **A**: Panoramica target (distribuzione, missingness, statistica descrittiva).
- **B**: Correlazioni numeriche — elenco feature più correlate (positive/negative) con il target.
- **C**: Scatter / regplot per le top feature correlate.
- **D**: Analisi categorica: boxplot di `estimated_occupancy_l365d` per `room_type` e per i top quartieri.
- **E**: Missingness per feature rilevanti.

In [ ]:
# A Target overview: `estimated_occupancy_l365d`
t = listings['estimated_occupancy_l365d']
print('Count non-null:', t.count())
print('Missing:', t.isna().sum())
display(t.describe())
plt.figure(figsize=(8,4))
sns.histplot(t.dropna(), kde=True, bins=50)
plt.title('Distribution of estimated_occupancy_l365d')
plt.xlabel('estimated_occupancy_l365d')
plt.show()
plt.figure(figsize=(6,3))
sns.boxplot(x=t)
plt.title('Boxplot of estimated_occupancy_l365d')
plt.show()

### Come leggere i grafici

- Istogramma + KDE: mostra la forma della distribuzione (asimmetria, picchi, code).
- Boxplot: mette in evidenza mediana, IQR e outlier.
- Statistiche rapide: media, mediana, quartili, count e missing.

### Cosa emerge

- Distribuzione fortemente asimmetrica con massa vicino a 0 e lunga coda verso valori alti: molti annunci hanno poche prenotazioni mentre pochi sono molto popolari (hotel, appartamenti centrali).
- Mediana relativamente bassa: la maggioranza degli annunci presenta occupazione contenuta.
- Presenza di outlier (valori molto alti) che vanno verificati ma non sono necessariamente errori.
- Potrebbe essere utile considerare trasformazioni del target o modelli robusti se si cerca stabilità nelle previsioni.


In [ ]:
# B Correlazioni con il target
exclude_prefixes = ['availability_', 'number_of_reviews']
exclude_exact = {'price', 'estimated_revenue_l365d'}
num_cols = listings.select_dtypes(include=[np.number]).columns.tolist()
if 'estimated_occupancy_l365d' in num_cols:
    num_cols.remove('estimated_occupancy_l365d')
cols_for_corr = [c for c in num_cols if c not in exclude_exact and not any(c.startswith(p) for p in exclude_prefixes)]
print(f'Total numeric features considered for correlation: {len(cols_for_corr)}')
corr_with_target = listings[cols_for_corr + ['estimated_occupancy_l365d']].corr()['estimated_occupancy_l365d'].drop('estimated_occupancy_l365d')
corr_sorted = corr_with_target.sort_values(ascending=False)
display(corr_sorted.head(20))
display(corr_sorted.tail(20))

### Come leggere

- La tabella ordina le feature per correlazione con `estimated_occupancy_l365d` (dal più positivo al più negativo).
- Guardare sia il segno (positivo/negativo) sia la magnitudo: le feature con valore assoluto più alto sono le più rilevanti.
- Sono state escluse le colonne di possibile leakage: `price`, `estimated_revenue_l365d`, `availability_*` e `number_of_reviews*`.

### Cosa emerge

- Le metriche di recensione (es. `reviews_per_month`, `review_scores_*`) mostrano le associazioni più evidenti.
- Gli effetti osservati sono generalmente moderati: le correlazioni non sono enormi e possono essere influenzate da confondenti o da differenze fra città.
- Usare queste indicazioni per selezionare le candidate feature da approfondire con scatterplot e analisi stratificate.


In [ ]:
# C Scatter / regplot per le top feature correlate
corr_vals = corr_with_target.abs().sort_values(ascending=False)
top_feats = corr_vals.head(5).index.tolist()
print('Top features by absolute correlation:', top_feats)

n_cols = 3
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 10))
axes = axes.ravel()

for ax, f in zip(axes, top_feats):
    sns.regplot(x=listings[f], y=listings['estimated_occupancy_l365d'], scatter_kws={'alpha':0.35}, ax=ax)
    ax.set_title(f'{f} vs estimated_occupancy_l365d')
    ax.set_xlabel(f)
    ax.set_ylabel('estimated_occupancy_l365d')

for ax in axes[len(top_feats):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

### Come leggere gli scatter/regplot

- Ogni grafico mostra la relazione tra una feature numerica e `estimated_occupancy_l365d`.
- I punti sparsi danno l'idea della variabilità reale dei dati, mentre la linea di regressione aiuta a capire se il legame è crescente, decrescente o quasi assente.
- Se la nuvola di punti è molto larga e la linea è quasi orizzontale, la feature è poco informativa da sola.
- La vista a scacchiera rende più facile confrontare i grafici e vedere subito quali relazioni sembrano più stabili.

### Cosa emerge

- Le feature più interessanti sono `reviews_per_month`, `latitude`, `longitude`, `review_scores_value` e `review_scores_accuracy`, ma il segnale non è sempre pulito.
- `reviews_per_month` sembra la più informativa, anche se la relazione è influenzata da alcuni valori molto estremi e da una forte concentrazione di punti nei valori bassi.
- Le variabili geografiche vanno lette con cautela: il dataset è aggregato per città, quindi `latitude` e `longitude` riflettono anche differenze tra mercati diversi.
- In generale, questi grafici servono soprattutto a individuare feature promettenti e a capire la forma della relazione, non a dire che una variabile è forte in senso assoluto.

In [ ]:
# D Analisi categorica (semplificata): room_type + neighbourhoods filtrati
min_count = 10

# Boxplot per `room_type`
plt.figure(figsize=(10,4))
order = listings['room_type'].value_counts().index
sns.boxplot(x='room_type', y='estimated_occupancy_l365d', data=listings, order=order)
plt.xticks(rotation=45)
plt.title('Occupancy by room_type (top categories)')
plt.show()

# Boxplot per `neighbourhood_cleansed` ma escludendo quartieri con pochi listing
counts = listings['neighbourhood_cleansed'].value_counts()
good_neigh = counts[counts >= min_count].index
print(f'Plotting neighbourhoods with >= {min_count} listings ({len(good_neigh)} neighbourhoods).')
top_to_plot = counts.loc[good_neigh].sort_values(ascending=False).head(15).index
if len(top_to_plot) > 0:
    plt.figure(figsize=(12,5))
    sns.boxplot(x='neighbourhood_cleansed', y='estimated_occupancy_l365d',
                data=listings[listings['neighbourhood_cleansed'].isin(top_to_plot)],
                order=top_to_plot)
    plt.xticks(rotation=45)
    plt.title(f'Occupancy by neighbourhood (min_count={min_count}) - top {len(top_to_plot)}')
    plt.show()

counts = listings['neighbourhood_cleansed'].value_counts()
filtered = counts[counts >= min_count].sort_values(ascending=False)
print(filtered.to_string())

### Come leggere i boxplot

- Ogni boxplot confronta la distribuzione di `estimated_occupancy_l365d` tra le categorie (es. `room_type` o `neighbourhood_cleansed`).
- La linea centrale è la mediana; il box mostra l'IQR (50% centrale). Baffi e punti fuori indicano variabilità e outlier.
- Attenzione alle categorie con pochi annunci: medie elevate possono dipendere da pochi valori estremi.
- In questa versione abbiamo applicato un filtro di supporto: vengono plottati solo i quartieri con almeno `min_count` listing (default 10). Controllare sempre il valore di `min_count` usato nella cella di codice.
- Controllare sempre la frequenza (count) per categoria insieme alla mediana prima di trarre conclusioni.

### Cosa emerge

- Alcuni quartieri presentano medie molto alte (es. Lenna, Pra' Secco, Ca' Brentelle): prima di considerare questi risultati, verificare il conteggio di listing per quei quartieri — se il conteggio è basso, la mediana o la media può essere fuorviante.
- `room_type` mostra differenze pratiche: intere case tendono ad avere mediane e variabilità maggiori rispetto a stanze condivise o private.
- Il filtro `min_count` riduce l'impatto dei quartieri con supporto scarso, rendendo le comparazioni più robuste; tuttavia potrebbe escludere nicchie reali se `min_count` è troppo alto.
- Questi grafici servono a individuare categorie e quartieri da approfondire con analisi stratificate o con filtri per numero di osservazioni; usare la stampa dei conteggi (ordinata per frequenza) per ispezionare i quartieri esclusi.


In [ ]:
# E Missingness summary per feature rilevanti
relevant = cols_for_corr.copy() if 'cols_for_corr' in globals() else []
relevant += ['estimated_occupancy_l365d'] if 'estimated_occupancy_l365d' in listings.columns else []
miss_pct = listings[relevant].isna().mean().sort_values(ascending=False)
print('Missingness percentage for relevant numeric features:')
display(miss_pct.head(20))
plt.figure(figsize=(6,8))
sns.heatmap(listings[relevant].isna().astype(int).sample(frac=0.2, random_state=42).T, cbar=False)
plt.title('Missingness heatmap (sampled rows)')
plt.show()

### Come leggere la missingness

- La tabella mostra la percentuale di valori mancanti per ciascuna feature: valori più alti indicano meno osservazioni utili per quella colonna.
- La heatmap campionata visualizza la struttura dei missing (colonne = variabili, righe = campione di record): strisce verticali indicano missing sparsi, blocchi orizzontali indicano assenze sistematiche su gruppi di righe.
- Confrontare sempre missingness e target: se il target è completo, le feature mancanti possono essere imputate o segnalate con indicatori di missing.
- Prestare attenzione a variabili derivate da recensioni (`review_scores_*`, `reviews_per_month`): spesso hanno missing non casuale (dipendono dall'assenza di recensioni).

### Cosa emerge

- Le feature legate alle recensioni mostrano missing non trascurabile: considerare imputazione (mediana) o l'uso di una variabile indicator per missingness.
- La heatmap suggerisce pattern sparsi piuttosto che blocchi grandi: i missing tendono a essere associati a singoli annunci, non a interi sottogruppi (ma verificare stratificando per città/quartiere).
- Regole pratiche consigliate: valutare l'eliminazione di colonne con percentuale di missing molto alta (es. >50%), oppure imputare con strategie stratificate; testare modelli con/senza le colonne imputate.
- Se i missing sono correlati a feature geografiche o categorie, preferire imputazione stratificata per evitare bias.


## Analisi esplorativa


In [ ]:
listings.head()

In [ ]:
listings.info()

Il dataset contiene 44 variabili. 

La variabile da predire per il primo task è la variabile `price`, la 12-esima del DataFrame.

Applico il metodo describe a listing per avere una prima idea della distribuzione dei dati.

In [ ]:
listings.describe()

Per il task di regressione su `price` è necessario filtrare i dati per rimuovere le feature non necessarie. In particolare è necessario rimuovere le feature riguardanti il tasso di occupazione, in quanto non sono importanti per la predizione del prezzo.

Creiamo quindi un nuovo DataFrame `lst_for_price_analysis` che contiene solo le feature necessarie per la predizione del prezzo. In particolare, rimuoviamo le feature riguardanti il tasso di occupazione e la disponibilità:

- `minimum_nights`, `maximum_nights`, `minimum_minimum_nights`, `maximum_minimum_nights`, `minimum_maximum_nights`, `maximum_maximum_nights`, `minimum_nights_avg_ntm`, `maximum_nights_avg_ntm`
- `calendar_updated`, `has_availability`, `availability_30`, `availability_60`, `availability_90`, `availability_365`, `calendar_last_scraped`
- `number_of_reviews`, `number_of_reviews_ltm`, `number_of_reviews_l30d`, `availability_eoy`, `number_of_reviews_ly`
- `estimated_occupancy_l365d`, `estimated_revenue_l365d`


Manteniamo quindi solo le feature che potrebbero essere utili per la predizione del prezzo, ovvero:
- `id`
- `listing_url`
- `neighbourhood_cleansed`
- `latitude`
- `longitude`
- `property_type`
- `room_type`
- `accommodates`
- `bathrooms`
- `bedrooms`
- `beds`
- `amenities`
- `price`

In [ ]:
filtered_features = [
    "id",
    "listing_url",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "amenities",
    "price"
]

lst_for_price_analysis = listings[filtered_features]

lst_for_price_analysis.info()

Il nuovo DataFrame `lst_for_price_analysis` contiene quindi 12 variabili, di cui 11 sono feature e 1 è la variabile da predire. Le feature sono sono perlopiù numeriche. Ci sono alcune feature categoriche:
- `neighbourhood_cleansed`
- `property_type`
- `room_type`
- `amenities`

Che possono essere importanti per la predizione del prezzo, ma che vanno trattate in modo diverso dalle feature numeriche. In particolare, è necessario trasformare le feature categoriche in nuove variabili binarie.
Inoltre bisogna trattare anche il valore di `price`, che è una stringa. Per poter utilizzare questa variabile e poterla visualizzare nei plot, è necessario trasformarla in un valore numerico.

Vediamo ora il formato della variabile `price`.

In [ ]:
print(lst_for_price_analysis["price"])

Come si può vedere, la variabile `price` è una stringa che contiene il simbolo del dollaro e le virgole per le migliaia. Per poter utilizzare questa variabile e poterla visualizzare nei plot, è necessario trasformarla in un valore numerico:

In [ ]:
lst_for_price_analysis["price"] = (
    lst_for_price_analysis["price"]
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
)
lst_for_price_analysis["price"]

Adesso `price` è una variabile numerica che può essere utilizzata per la predizione del prezzo.

Ora applichiamo il metodo describe al nuovo DataFrame `lst_for_price_analysis` per avere una prima idea della distribuzione dei dati dopo il filtraggio.

In [ ]:
lst_for_price_analysis.describe()

La `latitudine`, come aspettato, è compresa tra 35.6 e 46.5, mentre la `longitudine` tra 9 e 18.5, che corrispondono alla posizione geografica dell'Italia. Per quanto riguarda le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds`, si nota che sono presenti dei valori outliers.

Difatti il numero massimo di `accomodates` è 16, ancora accettabile, ma ben più alto dal valore del percentile 75 che equivale a 5. Per di più il numero massimo di `bathrooms` è 100, di `bedrooms` è 44 e di `beds` è 50, che sono valori molto elevati e potrebbero essere considerati outliers risultato di errori di immissione.

Questi valori si discostano significativamente dai dati, e possono influenzare negativamente la performance del modello di regressione. Per questo motivo, è necessario trattarli prima di procedere con la fase di modellazione.

Per quanto riguarda la variabile `price`, si nota che il prezzo massimo è 80000, mentre il prezzo del 75-esimo percentile è 168. Questo indica che ci sono dei valori di prezzo molto elevati che potrebbero anche essi essere considerati outliers.

Prima di andare a visualizzare con dei plot la distribuzione delle variabili, trattiamo i valori mancanti. Visualizziamo per ogni variabile il numero di valori mancanti:

In [ ]:
na_values = lst_for_price_analysis.isna().sum()
print(na_values)

Come possiamo vedere le variabili `bathrooms`, `bedrooms` e `beds` contengono un numero poco significativo di valori mancanti rispetto ai dati totali. Si suppone che questi valori mancanti siano dovuti a errori di immissione, siccome è impossibile avere un alloggio con 0 camere o 0 letti. Per quanto riguarda i bagni, è possibile che ci siano alloggi senza bagno, ma è più probabile che si tratti di errori di immissione. 

Per questi motivi:
- Per le variabili `bedrooms` e `beds`, si suppone che i valori mancanti siano dovuti a errori di immissione e verranno tolti dal dataset.
- Per la variabile `bathrooms`, si suppone che effettivamente ci siano alloggi senza bagno, quindi i valori mancanti verranno sostituiti con 0.

In [ ]:
lst_for_price_analysis = lst_for_price_analysis.dropna(subset=["bedrooms", "beds"])
lst_for_price_analysis["bathrooms"] = lst_for_price_analysis["bathrooms"].fillna(0)
na_values = lst_for_price_analysis.isna().sum()
print(na_values)

Ora non abbiamo più valori mancanti nelle variabili `bathrooms`, `bedrooms` e `beds`, e possiamo procedere con la visualizzazione della distribuzione delle variabili tramite dei plot.

Come primo plot, è possibile visualizzare la distribuzione della variabile `price` tramite un istogramma. In questo modo è possibile vedere se ci sono dei valori di prezzo molto elevati che potrebbero essere considerati outliers.

In [ ]:
lst_for_price_analysis["price"].plot.hist(
    bins=50,
    log=True,
)

Come si può vedere, la distribuzione della variabile `price` è molto sbilanciata, con la maggior parte dei valori concentrati tra 0 e 10000, con alcuni valori molto elevati che potrebbero essere considerati outliers. Questa cosa può essere vista anche con il boxplot:

In [ ]:
lst_for_price_analysis["price"].plot.box()

Tuttavia un numero così elevato potrebbe indicare la presenza di servizi di lusso, come ville o castelli, che potrebbero essere presenti nel dataset. Per questo motivo, è necessario analizzare più a fondo questi valori per capire se sono effettivamente outliers o se rappresentano una categoria di servizi di lusso.

Generiamo ora degli istogrammi per le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds` per visualizzare meglio la presenza di outliers.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds"]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel() # Appiattisce la matrice di assi in un array 1D per iterare più facilmente

for ax, col in zip(axes, cols):
    lst_for_price_analysis[col].dropna().plot.hist(
        bins=50,
        log=True,
        ax=ax,
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Come si può vedere bathrooms, bedrooms e beds presentano dei valori molto elevati che sono il risultato di probabili errori di immissione. Per questo motivo questi valori saranno eliminati dal dataset:
- `bathrooms` < 20
- `bedrooms` < 22
- `beds` < 35

In [ ]:
lst_wout_outl_p_a = lst_for_price_analysis[
    (lst_for_price_analysis["bathrooms"] < 20) &
    (lst_for_price_analysis["bedrooms"] < 22) &
    (lst_for_price_analysis["beds"] < 35)
]

Adesso generiamo i boxplot per tutte le variabili per visualizzare il risulato del filtraggio degli outliers dati da errori di immissione.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds", "price"]

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel() # Appiattisce la matrice di assi in un array 1D per iterare più facilmente

for ax, col in zip(axes, cols):
    lst_wout_outl_p_a[col].dropna().plot.box(
        ax=ax,
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Ora proviamo a visualizzare la relazione tra `price` e le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds` tramite dei scatter plot. In questo modo è possibile vedere se ci sono delle relazioni tra queste variabili e il prezzo, e se ci sono dei valori di prezzo molto elevati che potrebbero essere considerati outliers.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds"]
colors = ["blue", "orange", "green", "red"]

sample = lst_wout_outl_p_a.sample(n=5000, random_state=7112004)

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel()

for ax, col in zip(axes, cols):
    sample[["price", col]].plot.scatter(
        x=col,
        y="price",
        ax=ax,
        logy= True,
        c=colors[cols.index(col)]
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Price")

plt.tight_layout()
plt.show()

In tutti e quattro i grafici si vede una tendenza positiva, anche se debole: all'aumentare del numero di `accomodates`, `bathrooms`, `bedrooms` e `beds`, aumenta anche il prezzo. Sicuramente variabili categoriche che abbiamo lasciato fuori da questa analisi, come `neighbourhood_cleansed`, `property_type`, `room_type` e `amenities`, possono avere anche esse un impatto sul prezzo, e potrebbero essere utili per migliorare la performance del modello di regressione.

Adesso analizziamo queste variabili. Estraiamo i valori unici per ogni variabile categorica:

In [ ]:
neighbourhoods = lst_for_price_analysis["neighbourhood_cleansed"].value_counts()
property_types = lst_for_price_analysis["property_type"].value_counts()
room_types = lst_for_price_analysis["room_type"].value_counts()
amenities = lst_for_price_analysis["amenities"].value_counts()

In [ ]:
print(neighbourhoods.head())
neighbourhoods.describe()

In [ ]:
print(property_types.head())
print(property_types.describe())

In [ ]:
print(room_types.head())
print(room_types.describe())

In [ ]:
print(amenities.head())
amenities.describe()

Come si può vedere, la variabile `neighbourhood_cleansed` ha 1011 valori unici, `property_type` ha 118 valori unici, `room_type` ha 4 valori unici e `amenities` ha 183716 valori unici.

`property_type` e `room_type` hanno un numero di categorie alto, quindi verranno presi i top 50 più frequenti, mentre per `room_type` che ha solo 4 categorie le visualizzeremo tutte.

per quanto riguarda `amenities`, è una variabile molto complessa, in quanto contiene una lista di amenità per record. Per questo motivo, è necessario analizzare questa variabile in modo diverso rispetto alle altre variabili categoriche. Estraiamo tutte le amenità presenti nel dataset e contiamo la frequenza di ogni amenità. In questo modo è possibile vedere quali sono le amenità più frequenti e se ci sono delle amenità che potrebbero essere utili per la predizione del prezzo.

Proviamo ora a visualizzare la distribuzione di queste variabili categoriche tramite dei grafici a barre, per capire se ci sono delle categorie molto frequenti che potrebbero essere utili per il modello di regressione.

Adesso andremo a visualizzare la distribuzione delle amenities. Per fare questo, siccome le anemities sono una stringa che rappresenta una lista di stringhe, è necessario:
1. trasformare la stringa in una lista di stringhe
2. esplodere la lista in modo da avere una riga per ogni amenità
3. contare la frequenza di ogni amenità

In questo modo è possibile ottenere un risultato simile a quello delle altre variabili categoriche, con la frequenza di ogni categoria presente nel dataset.

In [ ]:
import ast
all_amenities = lst_for_price_analysis["amenities"].apply(ast.literal_eval).explode().value_counts()
all_amenities

Anche in questo caso, avenfo 15206 ameninities uniche, si visualizzeranno solo le 50 più frequenti, per avere un'idea della distribuzione delle amenità presenti nel dataset.

In [ ]:
top_neighbourhoods = neighbourhoods.head(50)
top_neighbourhoods.plot.bar(figsize=(14, 6), title="Top 50 Neighbourhoods")

Per quanto riguarda i quartieri, le categorie più frequenti sono i centri storici delle città più grandi, in quanto sono le zone più turistiche e quindi più richieste per l'affitto di case vacanze. Oltre ai centri storici, sono presenti anche quartieri residenziali e quartieri periferici, che potrebbero essere meno richiesti ma comunque presenti nel dataset. I livelli di granularità dei quartieri sono diversi, con alcuni quartieri che rappresentano intere città, mentre altri rappresentano solo dei quartieri specifici all'interno di una città.

In [ ]:
top_property_types = property_types.head(50)
top_property_types.plot.bar(figsize=(14, 6), title="Top 50 Property Types")

Si può vedere come le categorie più frequenti di `property_type` siano concentrate su appartamenti, case vacanze e case indipendenti, che sono le tipologie di alloggio più richieste per l'affitto di case vacanze. Oltre a queste tipologie, sono presenti anche altre tipologie di alloggio meno richieste ma comunque presenti nel dataset.

In [ ]:
top_amenities = all_amenities.head(50)
top_amenities.plot.bar(figsize=(14, 6), title="Top 50 Amenities")

Si può notare come le categorie hanno una distribuzione piuttosto uniforme, con nessuna categoria che rappresenta una percentuale eccessiva rispetto alle altre. Tuttavia è da considerare che questo campo contiene circa 16000 categorie uniche, quindi è possibile che alcune categorie siano rappresentate da un numero molto basso di istanze.

Nella parte del pre-processing, è possibile decidere di mantenere solo i servizi più frequenti, in modo da ridurre la dimensionalità del dataset e migliorare la performance del modello di regressione. Un altro dato utile potrebbe essere contare il numero di amenità presenti in ogni record, in modo da avere una variabile numerica che rappresenta la quantità di servizi offerti da ogni alloggio, che potrebbe essere ulteriormente utile per la predizione del prezzo.

In [ ]:
room_types.plot.bar(log=True)

Questo grafico mostra la distribuzione delle tipologie di alloggio presenti nel dataset. Come si può vedere, le tipologie più frequenti sono gli appartamenti, seguiti dalle stanze private. Il grafico è in scala logaritmica, in quanto ci sono solo un migliario di stanze d'hotel o stanze condivise, mentre per le altre tipologie di alloggio sono presenti decine di migliaia di istanze.

Le ultime variabili da esplorare che possono essere utili per la predizione del prezzo sono le densità di letti, camere e bagni per ogni alloggio. Aggiungiamo quindi quattro nuove variabili al dataset:
- `beds_per_person` = `beds` / `accomodates`: il numero di letti per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più confortevole, e quindi potrebbe essere associato a un prezzo più elevato.
- `bedrooms_per_person` = `bedrooms` / `accomodates`: il numero di camere da letto per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più spazioso, e quindi potrebbe essere associato a un prezzo più elevato.
- `bathrooms_per_person` = `bathrooms` / `accomodates`: il numero di bagni per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più confortevole, e quindi potrebbe essere associato a un prezzo più elevato.
- `beds_per_bedroom` = `beds` / `bedrooms`: il numero di letti per camera da letto. Un valore più alto di questa variabile potrebbe indicare un alloggio più spazioso, e quindi potrebbe essere associato a un prezzo più elevato.

In [ ]:
lst_w_densities = lst_wout_outl_p_a.copy()
lst_w_densities["beds_per_person"] = lst_w_densities["beds"] / lst_w_densities["accommodates"]
lst_w_densities["bedrooms_per_person"] = lst_w_densities["bedrooms"] / lst_w_densities["accommodates"]
lst_w_densities["bathrooms_per_person"] = lst_w_densities["bathrooms"] / lst_w_densities["accommodates"]
lst_w_densities["beds_per_bedroom"] = lst_w_densities["beds"] / lst_w_densities["bedrooms"]
print(lst_w_densities)

Ora creiamo un grafico nello stile di quello superiore, per visualizzare se ci sono delle relazioni tra queste nuove variabili e il prezzo.

In [ ]:
cols = ["beds_per_person", "bedrooms_per_person", "bathrooms_per_person", "beds_per_bedroom"]
sample = lst_w_densities.sample(n=7000, random_state=7112004)

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel()

for ax, col in zip(axes, cols):
    sample[["price", col]].plot.scatter(
        x=col,
        y="price",
        ax=ax,
        logy= True,
        logx= True,
        c=colors[cols.index(col)]
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Price")

plt.tight_layout()
plt.show()

Tutte e quattro le variabili non mostrano una relazione molto forte con il prezzo. Per una visualizzazione più chiara è stato utilizzato il logaritmo anche dell'asse x. Possono però essere effettuate le seguenti osservazioni:
- `beds_per_person` e `bedrooms_per_person`: si nota una lieve concentrazione dei prezzi più elevati in corrispondenza di valori prossimi a 1, ovvero listing con un letto o una camera per ospite. Tuttavia la dispersione è elevata, quindi il potere predittivo di queste feature è limitato.
- `bathrooms_per_person`: non emerge alcuna tendenza chiara. La distribuzione è sostanzialmente piatta lungo l'asse x, indicando una scarsa correlazione con il prezzo.
- `beds_per_bedroom`: si osserva una leggera tendenza inversa, infatti listing con più letti per camera (ipoteticamente dormitori o strutture condivise) tendono ad avere prezzi più bassi. Questa feature potrebbe quindi catturare indirettamente anche la tipologia di alloggio.